
# Boréal Marché — Association Rules with Apriori
**Course:** 420-975-VA — Unsupervised Learning  
**Goal:** Discover association patterns in shopping baskets without a target label, then use the mined rules for recommendations.

This notebook follows the project brief for Phases 1–4. The recommender is an **unsupervised learning** system because there is no labelled target variable; Apriori discovers frequent co-occurrence patterns directly from transaction data.

> Important: the assignment requires `mlxtend.frequent_patterns.apriori`. Do not replace Apriori with FP-Growth or another mining algorithm.


## Environment and imports

Dependencies are listed in `requirements.txt`. The notebook itself starts from the imports required for the analysis.

In [1]:

from pathlib import Path
import os
import warnings
import time
import threading
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import coo_matrix
from mlxtend.frequent_patterns import apriori, association_rules

ROOT = Path("..")
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)


# Keep notebook output clean; these warnings do not affect the calculations.
warnings.filterwarnings("ignore", category=FutureWarning)


## Phase 1 — Load and prepare

In [2]:

tx = pd.read_csv(
    DATA_DIR / "transactions.zip",
    usecols=["TransactionID", "Date", "Item"],
    parse_dates=["Date"]
)

products = pd.read_csv(DATA_DIR / "products.csv")

print("Rows:", f"{len(tx):,}")
print("Baskets:", f"{tx['TransactionID'].nunique():,}")
print("Products in transactions:", tx["Item"].nunique())
print("Products in catalogue:", len(products))
print("Average basket size:", round(len(tx) / tx["TransactionID"].nunique(), 2))
print("Date range:", tx["Date"].min().date(), "to", tx["Date"].max().date())


Rows: 8,379,837
Baskets: 1,000,000
Products in transactions: 98
Products in catalogue: 98
Average basket size: 8.38
Date range: 2026-01-01 to 2026-12-31



### Verified dataset checks

Using the provided files:

- Rows: **8,379,837**
- Unique baskets: **1,000,000**
- Unique products: **98**
- Average basket size: **8.38 items**
- Date range: **2026-01-01 to 2026-12-31**

These values should be reproduced by the code cell above when the notebook is run locally.


### Product frequencies

In [3]:

product_frequency = (
    tx.groupby("Item")["TransactionID"]
      .nunique()
      .sort_values(ascending=False)
)

display(product_frequency.head(10).rename("basket_count").to_frame())
display(product_frequency.tail(5).rename("basket_count").to_frame())


,basket_count
Item,
Sac réutilisable,900026
Biscuits,161439
Noix mélangées,155189
Beurre d'arachide,153726
Bière,150468
Savon à vaisselle,149820
Fromage cheddar,143457
Charbon de bois,140561
Papier hygiénique,139758


,basket_count
Item,
Safran,818
Huile de truffe,817
Caviar,810
Vinaigre balsamique vieilli,779
Foie gras,756



**Top 10 most frequent products**

- Sac réutilisable: **900,026** baskets
- Biscuits: **161,439** baskets
- Noix mélangées: **155,189** baskets
- Beurre d'arachide: **153,726** baskets
- Bière: **150,468** baskets
- Savon à vaisselle: **149,820** baskets
- Fromage cheddar: **143,457** baskets
- Charbon de bois: **140,561** baskets
- Papier hygiénique: **139,758** baskets
- Céréales: **139,198** baskets

**5 rarest products**

- Safran: **818** baskets
- Huile de truffe: **817** baskets
- Caviar: **810** baskets
- Vinaigre balsamique vieilli: **779** baskets
- Foie gras: **756** baskets



### Prediction before mining

`Sac réutilisable` is expected to cause trouble because it appears in an unusually large proportion of baskets.  
A highly prevalent consequent can make many rules look impressive by **confidence**, even when the antecedent adds little useful information.  
Therefore, before mining, the prediction is:

> Rules that recommend `Sac réutilisable` may rank very high by confidence but have lift close to 1.

This prediction will be tested in Phase 3 by comparing the confidence ranking with the lift ranking.


### Possible catalogue duplication

`Baguette` and `Pain baguette rustique` were inspected as possible duplicate catalogue entries because their names are similar.

The data provides evidence that they should **not** be merged:

- `Baguette` appears in **69,400** baskets.
- `Pain baguette rustique` appears in **42,219** baskets.
- They appear together in **3,009** baskets.
- Their catalogue prices are different: **$2.99** and **$3.99**.

Because both products can appear in the same basket and are priced differently, they are best treated as two separate SKUs rather than duplicate records. Merging them would risk artificially increasing support and changing the association structure.

**Decision:** keep them separate and document the duplicate-candidate check rather than modifying the source data.

In [4]:

# Evidence for the duplicate-candidate decision
candidate_a = "Baguette"
candidate_b = "Pain baguette rustique"

a_baskets = set(tx.loc[tx["Item"].eq(candidate_a), "TransactionID"])
b_baskets = set(tx.loc[tx["Item"].eq(candidate_b), "TransactionID"])

print(candidate_a, "basket count:", len(a_baskets))
print(candidate_b, "basket count:", len(b_baskets))
print("Baskets containing BOTH:", len(a_baskets & b_baskets))

display(products[products["Item"].isin([candidate_a, candidate_b])])


Baguette basket count: 69400
Pain baguette rustique basket count: 42219
Baskets containing BOTH: 3009


,Item,Item_EN,Category,Price
48,Baguette,Baguette,Boulangerie,2.99
96,Pain baguette rustique,Rustic baguette,Boulangerie,3.99


### Build the boolean basket × product matrix

In [5]:

# Memory-conscious construction:
# factorize basket IDs and product names, build a sparse boolean matrix,
# then expose it as a pandas SparseDataFrame accepted by mlxtend.

t0 = time.perf_counter()

basket_codes, basket_ids = pd.factorize(tx["TransactionID"], sort=False)
product_names = sorted(tx["Item"].unique())
product_to_col = {name: i for i, name in enumerate(product_names)}
product_codes = tx["Item"].map(product_to_col).to_numpy()

sparse_matrix = coo_matrix(
    (
        np.ones(len(tx), dtype=np.uint8),
        (basket_codes, product_codes)
    ),
    shape=(len(basket_ids), len(product_names))
).tocsr()

# Guarantee a boolean presence matrix even if duplicate rows exist
sparse_matrix.data[:] = 1
sparse_matrix = sparse_matrix.astype(bool)

basket = pd.DataFrame.sparse.from_spmatrix(
    sparse_matrix,
    index=basket_ids,
    columns=product_names
)

wide_seconds = time.perf_counter() - t0
wide_memory_mb = basket.memory_usage(deep=True).sum() / 1024**2

print("Basket shape:", basket.shape)
print("Build time (s):", round(wide_seconds, 2))
print("Pandas-reported memory (MB):", round(wide_memory_mb, 2))
print("Sparse density:", basket.sparse.density)


Basket shape: (1000000, 98)
Build time (s): 1.74
Pandas-reported memory (MB): 47.59
Sparse density: 0.08550854081632656


C:\Users\Somy\AppData\Local\Temp\ipykernel_22040\3416278069.py:24: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  basket = pd.DataFrame.sparse.from_spmatrix(



## Phase 2 — Mine frequent itemsets and association rules

The project requires at least four `min_support` runs: **0.05, 0.02, 0.01, 0.005**.

The helper below measures wall-clock time and approximates **peak process RSS** while Apriori is running.  
This is more useful for this project than `tracemalloc`, because NumPy/SciPy allocations may occur outside Python's own allocator.


In [6]:

def run_with_peak_rss(func, *args, sample_interval=0.05, **kwargs):
    process = psutil.Process(os.getpid())
    baseline = process.memory_info().rss
    peak = baseline
    stop = False

    def sampler():
        nonlocal peak, stop
        while not stop:
            try:
                peak = max(peak, process.memory_info().rss)
            except psutil.Error:
                pass
            time.sleep(sample_interval)

    thread = threading.Thread(target=sampler, daemon=True)
    thread.start()
    t0 = time.perf_counter()

    try:
        result = func(*args, **kwargs)
    finally:
        elapsed = time.perf_counter() - t0
        stop = True
        thread.join(timeout=1)

    peak_delta_mb = (peak - baseline) / 1024**2
    return result, elapsed, peak_delta_mb


In [7]:

MIN_CONFIDENCE = 0.30
SUPPORT_VALUES = [0.05, 0.02, 0.01, 0.005]

benchmark_rows = []
runs = {}

for min_support in SUPPORT_VALUES:
    print(f"\n=== min_support={min_support} ===")

    try:
        itemsets, apriori_time, apriori_peak_mb = run_with_peak_rss(
            apriori,
            basket,
            min_support=min_support,
            use_colnames=True,
            low_memory=True
        )

        rules = association_rules(
            itemsets,
            metric="confidence",
            min_threshold=MIN_CONFIDENCE
        )

        benchmark_rows.append({
            "min_support": min_support,
            "itemsets": len(itemsets),
            "rules": len(rules),
            "time_seconds": apriori_time,
            "peak_memory_mb_above_baseline": apriori_peak_mb,
            "outcome": "success"
        })

        runs[min_support] = {
            "itemsets": itemsets,
            "rules": rules
        }

        print("Itemsets:", len(itemsets))
        print("Rules:", len(rules))
        print("Time (s):", round(apriori_time, 2))
        print("Peak additional RSS (MB):", round(apriori_peak_mb, 2))

    except MemoryError:
        benchmark_rows.append({
            "min_support": min_support,
            "itemsets": np.nan,
            "rules": np.nan,
            "time_seconds": np.nan,
            "peak_memory_mb_above_baseline": np.nan,
            "outcome": "MemoryError"
        })
        print("MemoryError")

benchmark = pd.DataFrame(benchmark_rows)
display(benchmark)



=== min_support=0.05 ===
Itemsets: 191
Rules: 282
Time (s): 3.64
Peak additional RSS (MB): 212.36

=== min_support=0.02 ===
Itemsets: 658
Rules: 2707
Time (s): 7.52
Peak additional RSS (MB): 262.26

=== min_support=0.01 ===
Itemsets: 2267
Rules: 7817
Time (s): 15.86
Peak additional RSS (MB): 219.13

=== min_support=0.005 ===
Itemsets: 7106
Rules: 23918
Time (s): 44.94
Peak additional RSS (MB): 231.38


,min_support,itemsets,rules,time_seconds,peak_memory_mb_above_baseline,outcome
0,0.050,191,282,3.643036,212.355469,success
1,0.020,658,2707,7.521854,262.261719,success
2,0.010,2267,7817,15.859020,219.128906,success
3,0.005,7106,23918,44.942188,231.375000,success


### Benchmark interpretation

The latest full run produced the following measured results:

| min_support | Itemsets | Rules | Time (s) | Peak additional RSS (MB) |
|---:|---:|---:|---:|---:|
| 0.050 | 191 | 282 | 3.64 | 212.36 |
| 0.020 | 658 | 2,707 | 7.52 | 262.26 |
| 0.010 | 2,267 | 7,817 | 15.86 | 219.13 |
| 0.005 | 7,106 | 23,918 | 44.94 | 231.38 |

Lowering `min_support` greatly increases the search space. From 0.05 to 0.005, the number of frequent itemsets grows from 191 to 7,106 and the number of association rules grows from 282 to 23,918. Runtime increases from 3.64 seconds to 44.94 seconds.

Peak memory is not monotonic because process-level RSS depends on how intermediate arrays are allocated and released during each run. The largest measured peak in this run was **262.26 MB** at `min_support = 0.02`.

The final choice of `min_support = 0.005` is acceptable because mining is performed offline, while the online application only loads the saved rule file.


### Why `low_memory=True` matters

Apriori generates candidate itemsets and repeatedly checks their support. On a dataset with one million baskets, the intermediate candidate structures can become much larger than the final set of frequent itemsets.

`low_memory=True` changes the internal enumeration strategy so that fewer large intermediate arrays are held in memory at once. The trade-off is typically lower memory usage at the cost of additional computation.

For the report, keep the **actual local measurements** produced above. If a more aggressive run fails, document the failure rather than hiding it; the project brief explicitly values documented negative results.


## Phase 3 — Evaluate the rules


- **Support:** how often the complete item combination appears in all baskets.
- **Confidence:** among baskets containing the antecedent, how often the consequent also appears.
- **Lift:** how much more often antecedent and consequent occur together than we would expect from the consequent's overall popularity.

A lift of:
- `> 1` indicates a positive association,
- `≈ 1` indicates little or no association,
- `< 1` indicates a negative association.


In [8]:

# Use the most detailed successful run
chosen_support = min(runs.keys())
rules = runs[chosen_support]["rules"].copy()

simple_rules = rules[
    (rules["antecedents"].apply(len) == 1) &
    (rules["consequents"].apply(len) == 1)
].copy()

simple_rules["antecedent"] = simple_rules["antecedents"].apply(lambda x: next(iter(x)))
simple_rules["consequent"] = simple_rules["consequents"].apply(lambda x: next(iter(x)))

cols = ["antecedent", "consequent", "support", "confidence", "lift"]

print("Top 10 simple rules by confidence")
display(simple_rules.sort_values("confidence", ascending=False)[cols].head(10))

print("Top 10 simple rules by lift")
display(simple_rules.sort_values("lift", ascending=False)[cols].head(10))


Top 10 simple rules by confidence


,antecedent,consequent,support,confidence,lift
152,Concombre,Sac réutilisable,0.062853,0.902619,1.002881
141,Cidre de glace,Sac réutilisable,0.052520,0.902034,1.002231
324,Sauce tomate,Sac réutilisable,0.095588,0.901808,1.001980
306,Relish,Sac réutilisable,0.053301,0.901695,1.001854
228,Huile d'olive,Sac réutilisable,0.049201,0.901662,1.001818
298,Pommes,Sac réutilisable,0.082351,0.901617,1.001768
263,Moutarde de Dijon,Sac réutilisable,0.074374,0.901525,1.001665
208,Fromage suisse,Sac réutilisable,0.083997,0.901458,1.001592
330,Steak de boeuf,Sac réutilisable,0.095092,0.901414,1.001543
27,Avocat,Sac réutilisable,0.039907,0.901344,1.001464


Top 10 simple rules by lift


,antecedent,consequent,support,confidence,lift
88,Bouillon à fondue,Fromage à raclette,0.033651,0.696780,10.170786
89,Fromage à raclette,Bouillon à fondue,0.033651,0.491198,10.170786
133,Fromage à raclette,Charcuteries,0.031932,0.466106,9.975519
132,Charcuteries,Fromage à raclette,0.031932,0.683403,9.975519
157,Fromage à raclette,Cornichons,0.029733,0.434008,9.772967
156,Cornichons,Fromage à raclette,0.029733,0.669526,9.772967
210,Fromage à raclette,Pain baguette rustique,0.027627,0.403267,9.551784
209,Pain baguette rustique,Fromage à raclette,0.027627,0.654374,9.551784
14,Riz à sushi,Algues nori,0.054431,0.650738,9.504685
15,Algues nori,Riz à sushi,0.054431,0.795019,9.504685


### Confidence versus lift

The two rankings tell very different stories.

The confidence ranking is dominated by rules that recommend `Sac réutilisable`. Confidence is around 90% for many of these rules, but lift is approximately 1.00. This happens because `Sac réutilisable` already appears in about 90% of all baskets, so a high confidence value does not necessarily indicate a useful association.

The lift ranking surfaces more specific relationships, such as:

- `Bouillon à fondue ↔ Fromage à raclette`
- `Charcuteries ↔ Fromage à raclette`
- `Cornichons ↔ Fromage à raclette`
- `Riz à sushi ↔ Algues nori`

This confirms the Phase 1 prediction: a very common consequent can make confidence look impressive even when the antecedent adds little information.

For the production rule set, only rules with **lift > 1** are kept because the recommender should prioritize positive associations rather than neutral or negative ones.

In [9]:

filtered_rules = rules[rules["lift"] > 1].copy()

# Coverage = unique products that can appear as recommendations
recommendable_products = set()
for consequent in filtered_rules["consequents"]:
    recommendable_products.update(consequent)

coverage_count = len(recommendable_products)
coverage_pct = 100 * coverage_count / len(products)

print("Filtered rules:", len(filtered_rules))
print("Recommendable products:", coverage_count, "/", len(products))
print("Coverage:", round(coverage_pct, 2), "%")


Filtered rules: 22476
Recommendable products: 58 / 98
Coverage: 59.18 %


## Phase 4 — Seasonal analysis

In [10]:

summer_tx = tx[tx["Date"].dt.month.isin([6, 7, 8])].copy()
winter_tx = tx[tx["Date"].dt.month.isin([12, 1, 2])].copy()

print("Summer baskets:", summer_tx["TransactionID"].nunique())
print("Winter baskets:", winter_tx["TransactionID"].nunique())

def make_sparse_basket(frame, all_products):
    row_codes, row_ids = pd.factorize(frame["TransactionID"], sort=False)
    col_map = {name: i for i, name in enumerate(all_products)}
    col_codes = frame["Item"].map(col_map).to_numpy()

    mat = coo_matrix(
        (np.ones(len(frame), dtype=np.uint8), (row_codes, col_codes)),
        shape=(len(row_ids), len(all_products))
    ).tocsr()
    mat.data[:] = 1
    mat = mat.astype(bool)

    return pd.DataFrame.sparse.from_spmatrix(
        mat, index=row_ids, columns=all_products
    )

summer_basket = make_sparse_basket(summer_tx, product_names)
winter_basket = make_sparse_basket(winter_tx, product_names)

season_support = chosen_support

summer_itemsets = apriori(
    summer_basket, min_support=season_support,
    use_colnames=True, low_memory=True
)
winter_itemsets = apriori(
    winter_basket, min_support=season_support,
    use_colnames=True, low_memory=True
)

summer_rules = association_rules(
    summer_itemsets, metric="confidence",
    min_threshold=MIN_CONFIDENCE
)
winter_rules = association_rules(
    winter_itemsets, metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

print("Summer rules:", len(summer_rules))
print("Winter rules:", len(winter_rules))


Summer baskets: 252474
Winter baskets: 246965


C:\Users\Somy\AppData\Local\Temp\ipykernel_22040\2457140233.py:19: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  return pd.DataFrame.sparse.from_spmatrix(
C:\Users\Somy\AppData\Local\Temp\ipykernel_22040\2457140233.py:19: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  return pd.DataFrame.sparse.from_spmatrix(


Summer rules: 53842
Winter rules: 15396


In [11]:
def prepare_rules(df, season_name):
    out = df.copy()

    out["antecedent_key"] = out["antecedents"].apply(
        lambda x: tuple(sorted(x))
    )

    out["consequent_key"] = out["consequents"].apply(
        lambda x: tuple(sorted(x))
    )

    out = out[
        [
            "antecedent_key",
            "consequent_key",
            "support",
            "confidence",
            "lift"
        ]
    ].copy()

    out.columns = [
        "antecedent",
        "consequent",
        f"support_{season_name}",
        f"confidence_{season_name}",
        f"lift_{season_name}"
    ]

    return out


summer_cmp = prepare_rules(summer_rules, "summer")
winter_cmp = prepare_rules(winter_rules, "winter")

comparison = summer_cmp.merge(
    winter_cmp,
    on=["antecedent", "consequent"],
    how="inner"
)

comparison["lift_difference"] = (
    comparison["lift_summer"] - comparison["lift_winter"]
).abs()

comparison["confidence_difference"] = (
    comparison["confidence_summer"] -
    comparison["confidence_winter"]
).abs()

display(
    comparison.sort_values(
        "lift_difference",
        ascending=False
    ).head(20)
)

,antecedent,consequent,support_summer,confidence_summer,lift_summer,support_winter,confidence_winter,lift_winter,lift_difference,confidence_difference
9322,"(Ketchup, Pain à hamburger, Steak de boeuf)","(Charbon de bois, Sauce BBQ)",0.037671,0.485428,3.844838,0.005199,0.513600,29.979018,26.134179,0.028172
9328,"(Ketchup, Pain à hamburger)","(Charbon de bois, Sauce BBQ, Steak de boeuf)",0.037671,0.349348,3.838845,0.005199,0.365292,29.921805,26.082960,0.015944
9318,"(Charbon de bois, Sauce BBQ, Steak de boeuf)","(Ketchup, Pain à hamburger)",0.037671,0.413954,3.838845,0.005199,0.425871,29.921805,26.082960,0.011917
9327,"(Sauce BBQ, Steak de boeuf)","(Charbon de bois, Ketchup, Pain à hamburger)",0.037671,0.412804,3.837343,0.005199,0.414461,29.763695,25.926352,0.001657
9319,"(Charbon de bois, Ketchup, Pain à hamburger)","(Sauce BBQ, Steak de boeuf)",0.037671,0.350184,3.837343,0.005199,0.373364,29.763695,25.926352,0.023180
9323,"(Pain à hamburger, Sauce BBQ, Steak de boeuf)","(Charbon de bois, Ketchup)",0.037671,0.604103,3.850163,0.005199,0.621792,29.650667,25.800504,0.017689
9326,"(Ketchup, Steak de boeuf)","(Charbon de bois, Pain à hamburger, Sauce BBQ)",0.037671,0.331313,3.831260,0.005199,0.340313,29.614284,25.783024,0.009000
9320,"(Charbon de bois, Pain à hamburger, Sauce BBQ)","(Ketchup, Steak de boeuf)",0.037671,0.435625,3.831260,0.005199,0.452431,29.614284,25.783024,0.016806
9324,"(Ketchup, Sauce BBQ, Steak de boeuf)","(Charbon de bois, Pain à hamburger)",0.037671,0.684146,3.861163,0.005199,0.689581,29.281705,25.420542,0.005435
9317,"(Charbon de bois, Ketchup, Steak de boeuf)","(Pain à hamburger, Sauce BBQ)",0.037671,0.332146,3.830892,0.005199,0.346465,29.143316,25.312424,0.014319


In [12]:
summer_only = summer_cmp.merge(
    winter_cmp[["antecedent", "consequent"]],
    on=["antecedent", "consequent"],
    how="left",
    indicator=True
)

summer_only = summer_only[
    summer_only["_merge"] == "left_only"
].drop(columns="_merge")

display(
    summer_only.sort_values(
        "lift_summer",
        ascending=False
    ).head(20)
)

,antecedent,consequent,support_summer,confidence_summer,lift_summer
21509,"(Farine tout usage, Oeufs, Sac réutilisable)","(Bananes, Sirop d'érable)",0.005054,0.309558,16.847481
21510,"(Bananes, Farine tout usage)","(Oeufs, Sac réutilisable, Sirop d'érable)",0.005054,0.478620,16.839328
21506,"(Bananes, Sac réutilisable, Sirop d'érable)","(Farine tout usage, Oeufs)",0.005054,0.302513,16.756621
7334,"(Farine tout usage, Oeufs)","(Bananes, Sirop d'érable)",0.005517,0.305616,16.632942
7332,"(Bananes, Sirop d'érable)","(Farine tout usage, Oeufs)",0.005517,0.300280,16.632942
21504,"(Bananes, Farine tout usage, Sac réutilisable)","(Oeufs, Sirop d'érable)",0.005054,0.524240,16.615234
36978,"(Farine tout usage, Oeufs)","(Crème 35%, Sac réutilisable, Sirop d'érable)",0.005462,0.302545,16.569358
21226,"(Crème 35%, Farine tout usage)","(Bacon, Sac réutilisable, Sirop d'érable)",0.005054,0.436090,16.462536
21221,"(Crème 35%, Farine tout usage, Sac réutilisable)","(Bacon, Sirop d'érable)",0.005054,0.482055,16.455705
6794,"(Bananes, Farine tout usage)","(Bacon, Sirop d'érable)",0.005026,0.475994,16.248798


In [13]:
winter_only = winter_cmp.merge(
    summer_cmp[["antecedent", "consequent"]],
    on=["antecedent", "consequent"],
    how="left",
    indicator=True
)

winter_only = winter_only[
    winter_only["_merge"] == "left_only"
].drop(columns="_merge")

display(
    winter_only.sort_values(
        "lift_winter",
        ascending=False
    ).head(20)
)

,antecedent,consequent,support_winter,confidence_winter,lift_winter
13479,"(Charbon de bois, Sauce BBQ)","(Ketchup, Pain à hamburger, Steak de boeuf)",0.005199,0.303474,29.979018
13480,"(Pain à hamburger, Steak de boeuf)","(Charbon de bois, Ketchup, Sauce BBQ)",0.005199,0.301267,29.074798
13469,"(Charbon de bois, Pain à hamburger, Steak de boeuf)","(Ketchup, Sauce BBQ)",0.005199,0.307545,28.542965
10266,"(Pain à hamburger, Steak de boeuf)","(Ketchup, Sauce BBQ)",0.005199,0.301267,27.960318
8299,"(Charbon de bois, Pain à hamburger)","(Ketchup, Sauce BBQ)",0.007090,0.301066,27.941665
13515,"(Charbon de bois, Sac réutilisable, Saucisses)","(Ketchup, Sauce BBQ)",0.005171,0.300966,27.932409
8373,"(Charbon de bois, Saucisses)","(Ketchup, Sauce BBQ)",0.005758,0.300825,27.919300
14663,"(Gingembre mariné, Wasabi)","(Algues nori, Avocat, Riz à sushi, Sac réutilisable)",0.005624,0.303408,15.639962
14802,"(Algues nori, Gingembre mariné)","(Riz à sushi, Sac réutilisable, Sauce soya, Wasabi)",0.008507,0.304140,15.615793
14630,"(Gingembre mariné, Sauce soya)","(Algues nori, Avocat, Riz à sushi, Sac réutilisable)",0.007147,0.301813,15.557743


### Seasonal analysis

The seasonal mining results show a substantial difference between summer and winter purchasing patterns.

- Summer baskets: **252,474**
- Winter baskets: **246,965**
- Summer rules: **53,842**
- Winter rules: **15,396**

A useful shared example is:

**(Ketchup, Pain à hamburger, Steak de boeuf) → (Charbon de bois, Sauce BBQ)**

- Summer support: **0.0377**
- Summer confidence: **0.4854**
- Summer lift: **3.84**
- Winter support: **0.0052**
- Winter confidence: **0.5136**
- Winter lift: **29.98**

The rule is much more common in summer, but its lift is dramatically higher in winter because the consequent combination is much rarer during winter. This shows why lift is essential for interpreting association strength relative to baseline popularity.

A summer-only example is:

**(Farine tout usage, Oeufs, Sac réutilisable) → (Bananes, Sirop d'érable)**

In winter, this rule has support **0.0047** and confidence **0.2981**, so it falls below both thresholds (`min_support = 0.005`, `min_confidence = 0.30`).

A winter-only example is:

**(Charbon de bois, Sauce BBQ) → (Ketchup, Pain à hamburger, Steak de boeuf)**

In summer, its support is **0.0377**, but confidence is **0.2984**, just below the required 0.30 threshold.

These results show that a rule can disappear from a seasonal rule set because it no longer meets the selected thresholds, not necessarily because the item combination never occurs.

**Recommendation:** Boréal Marché should re-mine its rules at least quarterly, and preferably before major seasonal periods, because a single year-round rule set can become less relevant when shopping behaviour changes.

In [14]:
def fast_rule_metrics(basket_df, antecedent, consequent):
    ant_cols = list(antecedent)
    cons_cols = list(consequent)

    # فقط ستون‌های لازم را dense می‌کنیم، نه کل دیتاست
    ant_arr = basket_df[ant_cols].sparse.to_dense().to_numpy(dtype=bool)
    cons_arr = basket_df[cons_cols].sparse.to_dense().to_numpy(dtype=bool)

    ant_mask = ant_arr.all(axis=1)
    cons_mask = cons_arr.all(axis=1)
    both_mask = ant_mask & cons_mask

    support = both_mask.mean()
    ant_support = ant_mask.mean()
    cons_support = cons_mask.mean()

    confidence = support / ant_support if ant_support > 0 else 0
    lift = confidence / cons_support if cons_support > 0 else 0

    return support, confidence, lift


# Rule present in summer, checked in winter
rule1_ant = ("Farine tout usage", "Oeufs", "Sac réutilisable")
rule1_cons = ("Bananes", "Sirop d'érable")

r1 = fast_rule_metrics(
    winter_basket,
    rule1_ant,
    rule1_cons
)

print("Summer-only rule checked in WINTER")
print("Support:", r1[0])
print("Confidence:", r1[1])
print("Lift:", r1[2])


# Rule present in winter, checked in summer
rule2_ant = ("Charbon de bois", "Sauce BBQ")
rule2_cons = ("Ketchup", "Pain à hamburger", "Steak de boeuf")

r2 = fast_rule_metrics(
    summer_basket,
    rule2_ant,
    rule2_cons
)

print("\nWinter-only rule checked in SUMMER")
print("Support:", r2[0])
print("Confidence:", r2[1])
print("Lift:", r2[2])

Summer-only rule checked in WINTER
Support: 0.004717267629016257
Confidence: 0.2981064483111566
Lift: 16.671616622999274

Winter-only rule checked in SUMMER
Support: 0.037671205747918594
Confidence: 0.29837495294265276
Lift: 3.8448383539653608


## Phase 5 — Save the rules for the online application

The mining step is performed offline. The Streamlit application loads the saved rule and metadata files and never runs Apriori itself.

In [15]:
import json
from datetime import datetime

# Use the chosen final rule set
final_rules = runs[0.005]["rules"].copy()

# Keep only positive associations
final_rules = final_rules[final_rules["lift"] > 1].copy()

def frozenset_to_list(value):
    return sorted(list(value))

rules_export = []

for _, row in final_rules.iterrows():
    rules_export.append({
        "antecedents": frozenset_to_list(row["antecedents"]),
        "consequents": frozenset_to_list(row["consequents"]),
        "support": float(row["support"]),
        "confidence": float(row["confidence"]),
        "lift": float(row["lift"]),
        "antecedent_support": float(row["antecedent support"]),
        "consequent_support": float(row["consequent support"])
    })

rules_path = MODEL_DIR / "rules.json"

with open(rules_path, "w", encoding="utf-8") as f:
    json.dump(
        rules_export,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved rules:", len(rules_export))
print("Path:", rules_path)

Saved rules: 22476
Path: ..\models\rules.json


In [16]:
recommendable_products = set()

for rule in rules_export:
    recommendable_products.update(rule["consequents"])

coverage_count = len(recommendable_products)
coverage_pct = 100 * coverage_count / len(products)

metadata = {
    "algorithm": "Apriori",
    "library": "mlxtend",
    "min_support": 0.005,
    "min_confidence": 0.30,
    "min_lift": 1.0,
    "mining_date": datetime.now().isoformat(timespec="seconds"),
    "number_of_rules": len(rules_export),
    "catalogue_products": int(len(products)),
    "recommendable_products": int(coverage_count),
    "coverage_percent": float(round(coverage_pct, 2)),
    "offline_mining": True
}

metadata_path = MODEL_DIR / "metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print(json.dumps(metadata, indent=2))
print("Saved:", metadata_path)

{
  "algorithm": "Apriori",
  "library": "mlxtend",
  "min_support": 0.005,
  "min_confidence": 0.3,
  "min_lift": 1.0,
  "mining_date": "2026-10-29T20:26:37",
  "number_of_rules": 22476,
  "catalogue_products": 98,
  "recommendable_products": 58,
  "coverage_percent": 59.18,
  "offline_mining": true
}
Saved: ..\models\metadata.json


## Phase 6 — Measure and document

The measured Apriori benchmark is saved to the reports folder for reproducibility and inclusion in the final report.

## Save benchmark table

In [17]:

benchmark.to_csv(REPORT_DIR / "benchmark_results.csv", index=False)
benchmark


,min_support,itemsets,rules,time_seconds,peak_memory_mb_above_baseline,outcome
0,0.050,191,282,3.643036,212.355469,success
1,0.020,658,2707,7.521854,262.261719,success
2,0.010,2267,7817,15.859020,219.128906,success
3,0.005,7106,23918,44.942188,231.375000,success


## Final summary

- Algorithm: **Apriori** (`mlxtend`)
- Final `min_support`: **0.005**
- Minimum confidence: **0.30**
- Final rule filter: **lift > 1**
- Frequent itemsets at `min_support = 0.005`: **7,106**
- Rules before lift filtering: **23,918**
- Saved positive-association rules: **22,476**
- Catalogue coverage: **58 / 98 products = 59.18%**
- Summer rules: **53,842**
- Winter rules: **15,396**
- Mining is performed **offline**; the Streamlit application loads the saved JSON files and performs fast recommendation matching online.

The results also demonstrate two important interpretation lessons:

1. High confidence can be misleading when the consequent is already very common.
2. Association strength is seasonal, so rule sets should be re-mined periodically.